# Salesforce OAuth2 Integration with AI Lead Scoring
## Overview
This notebook demonstrates a live Salesforce integration using OAuth2 authentication.
It creates leads via Python, fetches them from Salesforce, scores them using the 
AI Lead Scoring Agent, and writes the scores back to Salesforce in real time.

## Tech Stack
- Python + Jupyter Notebook
- Salesforce (OAuth2 Client Credentials Flow)
- Anthropic Claude API (claude-haiku)
- simple-salesforce library

In [1]:
!pip install anthropic python-dotenv simple-salesforce -q

In [2]:
from dotenv import load_dotenv
import os
import anthropic
import requests
import re
from simple_salesforce import Salesforce
from scoring_config import scoring_criteria

load_dotenv()
client = anthropic.Anthropic()

## Step 1: Authenticate to Salesforce
Using OAuth2 Client Credentials flow — no username/password required.

In [3]:
auth_url = 'https://orgfarm-4a8f4d6146-dev-ed.develop.my.salesforce.com/services/oauth2/token'

response = requests.post(auth_url, data={
    'grant_type': 'client_credentials',
    'client_id' : os.getenv("SF_CONSUMER_KEY"),
    'client_secret' : os.getenv("SF_CONSUMER_SECRET"),
}
                       )
auth = response.json()
print ("Auth Status:", response.status_code)

Auth Status: 200


## Step 2: Connect to Salesforce

In [4]:
access_token = auth['access_token']
instance_url = auth['instance_url']

sf = Salesforce(instance_url=instance_url, session_id=access_token)
print("Connected:", sf.base_url)

Connected: https://orgfarm-4a8f4d6146-dev-ed.develop.my.salesforce.com/services/data/v59.0/


## Step 3: Define Leads
Define the list of leads to be created in Salesforce.

In [5]:
leads = [
    {
        'FirstName': 'John',
        'LastName': 'Smith',
        'Company': 'TechCorp Inc',
        'Email': 'john.smith@techcorp.com',
        'Title': 'VP of Marketing',
        'Industry': 'Technology',
        'AnnualRevenue': 5000000,
        'NumberOfEmployees': 500
    },
    {
        'FirstName': 'Sarah',
        'LastName': 'Johnson',
        'Company': 'FinanceHub',
        'Email': 'sarah.j@financehub.com',
        'Title': 'CFO',
        'Industry': 'Finance',
        'AnnualRevenue': 10000000,
        'NumberOfEmployees': 1000
    },
    {
        'FirstName': 'Mike',
        'LastName': 'Patel',
        'Company': 'StartupXYZ',
        'Email': 'mike@startupxyz.com',
        'Title': 'Founder',
        'Industry': 'Technology',
        'AnnualRevenue': 500000,
        'NumberOfEmployees': 20
    },
    {
        'FirstName': 'Lisa',
        'LastName': 'Chen',
        'Company': 'RetailGiant',
        'Email': 'lisa.chen@retailgiant.com',
        'Title': 'Director of Operations',
        'Industry': 'Retail',
        'AnnualRevenue': 50000000,
        'NumberOfEmployees': 5000
    },
    {
        'FirstName': 'David',
        'LastName': 'Kumar',
        'Company': 'HealthPlus',
        'Email': 'david.k@healthplus.com',
        'Title': 'CTO',
        'Industry': 'Healthcare',
        'AnnualRevenue': 2000000,
        'NumberOfEmployees': 150
    }
]

# for lead in leads:
#    result = sf.Lead.create(lead)
#   print(f"Created lead: {lead['FirstName']} {lead['LastName']} - ID: {result['id']}")

## Step 4: Create Leads in Salesforce
Checks for duplicates before creating — safe to run multiple times.

In [6]:
def create_lead_if_not_exists(lead_data):
    # Check if lead already exists
    query = f"SELECT Id FROM Lead WHERE Email = '{lead_data['Email']}'"
    result = sf.query(query)
    
    if result['totalSize'] > 0:
        print(f"Lead already exists: {lead_data['FirstName']} {lead_data['LastName']} - skipping")
        return result['records'][0]['Id']
    else:
        result = sf.Lead.create(lead_data)
        print(f"Created lead: {lead_data['FirstName']} {lead_data['LastName']} - ID: {result['id']}")
        return result['id']

for lead in leads:
    create_lead_if_not_exists(lead)

Lead already exists: John Smith - skipping
Lead already exists: Sarah Johnson - skipping
Lead already exists: Mike Patel - skipping
Lead already exists: Lisa Chen - skipping
Lead already exists: David Kumar - skipping


## Step 5: Fetch Leads from Salesforce
Query all leads leads in the ORG using SOQL. 

In [7]:
result = sf.query("SELECT Id, FirstName, LastName, Company, Title, Industry, AnnualRevenue, NumberOfEmployees FROM Lead")

leads_from_sf = result['records']

for lead in leads_from_sf:
    print(f"{lead['FirstName']} {lead['LastName']} - {lead['Company']} - {lead['Title']}")

Bertha Boxer - Farmers Coop. of Florida - Director of Vendor Relations
Phyllis Cotton - Abbott Insurance - CFO
Jeff Glimpse - Jackson Controls - SVP, Procurement
Mike Braund - Metropolitan Health Services - VP, Technology
Patricia Feager - International Shipping Co. - CEO
Brenda Mcclure - Cadinal Inc. - CFO
Violet Maccleod - Emerson Transport - VP, Finance
Kathy Snyder - TNR Corp. - Regional General Manager
Tom James - Delphi Chemicals - SVP, Production
Shelly Brownell - Western Telecommunications Corp. - SVP, Technology
Pamela Owenby - Hendrickson Trading - SVP, Technology
Norm May - Greenwich Media - VP, Facilities
Pat Stumuller - Pyramid Construction Inc. - SVP, Administration and Finance
Andy Young - Dickenson plc - SVP, Operations
Kristen Akin - Aethna Home Products - Director, Warehouse Mgmt
David Monaco - Blues Entertainment Corp. - CFO
Carolyn Crenshaw - Ace Iron and Steel Inc. - VP, Technology
Bill Dadio Jr - Zenith Industrial Partners - CFO
Jack Rogers - Burlington Textiles C

## Step 6: Fetch Today's Leads from Salesforce
Query only leads created today using SOQL — filters out sample data.- WHERE CreatedDate = TODAY

In [8]:
result = sf.query("SELECT Id, FirstName, LastName, Company, Title, Industry, AnnualRevenue, NumberOfEmployees FROM Lead WHERE CreatedDate = TODAY")

leads_from_sf = result['records']

for lead in leads_from_sf:
    print(f"{lead['FirstName']} {lead['LastName']} - {lead['Company']} - {lead['Title']}")

John Smith - TechCorp Inc - VP of Marketing
Sarah Johnson - FinanceHub - CFO
Mike Patel - StartupXYZ - Founder
Lisa Chen - RetailGiant - Director of Operations
David Kumar - HealthPlus - CTO


## Step 7: AI Lead Scoring + Write Back to Salesforce
Score each lead using the Lead Scoring Agent and write the AI score back as a custom field.

In [9]:
def extract_score(text):
    match = re.search(r'SCORE:\s*(\d+)/100', text)
    if match:
        return int(match.group(1))
    return None

for lead in leads_from_sf:
    sf_lead = {
        "name": f"{lead['FirstName']} {lead['LastName']}",
        "title": lead['Title'],
        "company": lead['Company'],
        "employees": lead['NumberOfEmployees'],
        "industry": lead['Industry'],
        "annual_revenue": lead['AnnualRevenue']
    }
    
    print(f"\n{'='*50}")
    print(f"Scoring lead: {sf_lead['name']} - {sf_lead['title']}")
    print(f"{'='*50}")
    
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": f"You are a MarTech Lead scoring expert. Use this {scoring_criteria} and score this lead:\n{sf_lead}"
            }
        ]
    )
    
    response_text = message.content[0].text
    print(response_text)
    
    score = extract_score(response_text)
    sf.Lead.update(lead['Id'], {'AI_Lead_Score__c': score})
    print(f"✓ Score {score} written to Salesforce")


Scoring lead: John Smith - VP of Marketing
SCORE: 80/100

BREAKDOWN:
- **Job Title (25/25 points):** VP of Marketing = VP level = 25 points
- **Company Size (20/20 points):** 500+ employees = 20 points
- **Intent Signal (0/25 points):** No intent signal provided in lead data = 0 points
- **Engagement (15/30 points):** Email opened = 5 points + Form submitted = 5 points + Website visits 5-9 = 15 points (estimated based on typical engagement) = 15 points*

*Note: Engagement details not explicitly provided in lead data; scored conservatively at 15 points based on assumed moderate engagement.

RECOMMENDATION: **WARM**

NEXT ACTION: Schedule a discovery call with John Smith within 24 hours to understand TechCorp's current marketing challenges and establish product-market fit alignment. Personalize outreach referencing his VP-level responsibilities and company scale.
✓ Score 80 written to Salesforce

Scoring lead: Sarah Johnson - CFO
SCORE: 80/100

BREAKDOWN:
- **Job Title (25/25):** CFO is